In [ ]:
%pip uninstall -y transformers deepchem
%pip install -U "transformers<5" safetensors genomic-benchmarks
%pip install -U "deepchem[torch] @ git+https://github.com/deepchem/deepchem.git"

In [ ]:
%pip uninstall -y torchao

In [ ]:
!git clone https://github.com/hanara2112/genomic_benchmark.git
%cd genomic_benchmark
!pip install -e . -q

In [ ]:
!git pull

In [1]:
from genomics.loader import load_genomic_benchmark
from genomics.dnabert2 import DNABERT2Model
import deepchem as dc

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
2026-03-13 23:10:13.546863: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773443413.568300     266 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773443413.575043     266 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin

Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (/usr/local/lib/python3.12/dist-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'haiku'


In [2]:
tasks, datasets, transformers = load_genomic_benchmark(
    dataset_name="human_nontata_promoters",
    splitter="random",
    reload=False,
)
train, valid, test = datasets

In [3]:
print(tasks)
print(len(train), len(valid), len(test))
print(type(train.X[0]), train.X[0][:50], train.y[0])

['human_nontata_promoters']
28904 3613 3614
<class 'str'> AGCCCCGCCCGCTTATCTGCCATTTAAGCCAGCGCCCCTTGAGAATGGAC [1.]


In [4]:
import deepchem as dc

metrics = [
    dc.metrics.Metric(dc.metrics.roc_auc_score),
    dc.metrics.Metric(dc.metrics.accuracy_score),
    dc.metrics.Metric(dc.metrics.f1_score),
]

In [7]:
print(train.X[:2])
print(train.y[:2])

['AGCCCCGCCCGCTTATCTGCCATTTAAGCCAGCGCCCCTTGAGAATGGACTACAACTCCCACAATGCATTCGAGGGGCACCGTGTTAGGGCGCCACCTTTTTCCCGGCTGTGCTGTACCTACAGCACCCAGAACTACAGTACCCAGAAAGCTCCGCTTCCCCTCCTGGGTGTTATTGCCCCTTTCCTGGTGCAGGACGCGGTAGTGGCCAGCGAGAGTGTCAGGCCTGGGGTTTTCTGTGTCCTTCCCTGG'
 'ATGGTCTTCCCGGCTTGGGAAAGGAATGAGAAGCTTTCATGATTCTCTCGCGAGAAGGGAAGTCTTCATGCCACGTCAGAGACTAGAGATCCTGGCGCCGAGCGACAGGTGCCGGAACAAACAGGCGATGAGAAATCGCGGACGCGGAACCATTTCTTGGGCAGGACTTCCGGCGGAAAAGCGGGCTGTCTCGGAAACTCAGAGCCGGGTTCCTCCCGGGTTTCTGCCGGGTTTCTCCCTGCGGCTCCTGG']
[[1.]
 [1.]]


In [ ]:
from genomics.dnabert2 import DNABERT2Model

for epoch in [1, 2, 3, 5]:
    model = DNABERT2Model(
        task="classification",
        n_tasks=1,
        max_seq_length=256,
        model_dir=f"./dnabert2_ckpt_{epoch}",
    )
    model.fit(train, nb_epoch=epoch)
    valid_scores = model.evaluate(valid, metrics)
    print(f"epoch={epoch}", valid_scores)


epoch=1 {'roc_auc_score': 0.5408874071664769, 'accuracy_score': 0.5358427899252699, 'f1_score': 0.6977833843935844}
epoch=2 {'roc_auc_score': 0.4912551991701041, 'accuracy_score': 0.5358427899252699, 'f1_score': 0.6977833843935844}
epoch=3 {'roc_auc_score': 0.49151970078899254, 'accuracy_score': 0.5358427899252699, 'f1_score': 0.6977833843935844}


In [6]:
valid_scores = model.evaluate(valid, metrics)
test_scores = model.evaluate(test, metrics)

print("valid:", valid_scores)
print("test:", test_scores)

valid: {'roc_auc_score': 0.5067245782758468, 'accuracy_score': 0.5358427899252699, 'f1_score': 0.6977833843935844}
test: {'roc_auc_score': 0.50764986721847, 'accuracy_score': 0.5437188710570006, 'f1_score': 0.7044273167234272}
